---
# 6. Model Evaluation

In this section, we will be evaluating all the GAN Models that we have trained in the above sections. Subsequently, we will be identifying the best performing GAN Model and creating a Conditional Version of it so as to evaluate if the performance will improve for that specific GAN Model as compared to a unconditional variant. The Model Evaluation will be conducted in this section.

---
## 6.1 Model Metrics Evaluation

In this sub-section, we will be evaluating all the GAN Models trained inclusive of Augmented and Non-Augmented Training Data Models. We will be using Kernel Inception Distance to serve as the main evaluation metrics due to the absence of bias within the KID Score. The relevant mathematical formulas for the various quantitative metrics are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Evalution in the Code Cell below.

In [ ]:
# =========== Function to Build and Sort GAN Results Table =========== #
def build_gan_results_table(model_names):
    records = []

    for model in model_names:
        prefix = model.upper()

        # ----- Safely Fetch Global Variables using Prefix-based Keys ----- #
        record = {
            "Model": model,
            "FID Score": globals().get(f"{prefix}_FID"),
            "KID Score": globals().get(f"{prefix}_KID"),
            "Mean Generator Loss": globals().get(f"{prefix}_GEN_LOSS_MEAN"),
            "Std Generator Loss": globals().get(f"{prefix}_GEN_LOSS_STD"),
            "Mean Discriminator Loss": globals().get(f"{prefix}_DISC_LOSS_MEAN"),
            "Std Discriminator Loss": globals().get(f"{prefix}_DISC_LOSS_STD"),
            "Mode Collapse Risk": globals().get(f"{prefix}_MODE_COLLAPSE"),
            "Diversity (t-SNE Spread)": globals().get(f"{prefix}_TSNE"),
            "PPL": globals().get(f"{prefix}_PPL")
            }

        # ----- Only Include Models with FID Computed ----- #
        if record["FID Score"] is not None:
            records.append(record)

    df = pd.DataFrame(records)

    # ----- Sort by KID Score ----- #
    df_sorted = df.sort_values(
        by=["FID Score", "KID Score"],
        ascending=[True, True]
    ).reset_index(drop=True)

    return df_sorted

# =========== Model Names =========== #
model_names = ["Vanilla GAN (Non-Augmented)", "DCGAN (Non-Augmented)", "WGAN (Non-Augmented)", "LSGAN (Non-Augmented)", "SAGAN (Non-Augmented)", "InfoGAN (Non-Augmented)", "DRAGAN (Non-Augmented)", "WGAN-GP (Non-Augmented)", "Vanilla GAN (Augmented)", "DCGAN (Augmented)", "WGAN (Augmented)", "LSGAN (Augmented)", "SAGAN (Augmented)", "InfoGAN (Augmented)", "DRAGAN (Augmented)", "WGAN-GP (Augmented)"]

# =========== Create DataFrame =========== #
df_all_gans = build_gan_results_table(model_names)

# =========== Display DataFrame =========== #
df_all_gans.style.background_gradient(cmap="Blues")

With reference to the output above, we are able to determine that the best model with respect to FID and KID is DRAGAN (Non-Augmented). We will subsequently be creating a Conditional-DRAGAN (cDRAGAN) in the next sub-section.

---
## 6.2 Conditional Deep Regret Analytic GAN Model Training

In this sub-section, we will be training a Conditional Deep Regret Analytic GAN (cDRAGAN) for the EMNIST Dataset. The LSGAN mainly replace the Binary Crossentropy Loss used in classical GANs with a Least Squares Loss. Through this replacement, the LSGAN is more likely able to have a Stabalise GAN Training, Produce Higher Quality Images, and Encourage the Generator more Accurate Results. The relevant mathematical formulas are indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = -\mathbb{E}_{z \sim p_z(z)} [\log D(G(z))]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Pertubed Input Sampling:**

Purpose: Add noise to real images to compute gradient penalty near the data manifold.

$$
\hat{x} = x + \alpha \cdot \sigma_x \cdot \epsilon
$$
Where:
- $x$ = Real Data Sample
- $\alpha \sim \mathcal{U}(0, 1)$ = Uniform Random Variable
- $\epsilon \sim \mathcal{N}(0, 1)$ = Standard Normal Noise
- $\sigma_x$ = Standard Deviation of Data Batch (per pixel)

---
**Minimax Objective with DRAGAN Penalty:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{data}} [\log D(x)] + \mathbb{E}_{z \sim p_z(z)} [\log(1 - D(G(z)))] - \lambda \cdot \mathbb{E}_{\hat{x} \sim p_{perturbed}} \left[ \left( \|\nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$
Where:
- $V(D, G)$ = Value Function for Adversarial Training
---
With the relevant mathematical formulas indicated above, we will proceed to train the LSGAN in this sub-section for the purpose of it's potential for smoother training process and also to serve as a comparison for our other GAN Models.

---
### 6.2.1 Defining Data Pre-Processing Function

In this sub-section, we will be defining the Data Pre-Processing Function which will allow for consistent Data Pipeline to be built and modifed accordingly should the need arise such as when we need to add Spectral Normalisation. This will ensure that the Data's integrity is not affected and we are able to make fair and un-biased assumptions. The Data Pre-Processing Function is defined in the Code Cell below:

In [ ]:
# ========== Data Pre-Processing Function for cDRAGAN ==========
def load_data(X_train, y_train, batch_size=32):
    buffer_size = X_train.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    dataset = dataset.shuffle(buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

With reference to the Code Cell above, we are able to determine that the Data Pre-Processing Function have been defined successfully and we are able to move on to Defining Callback Functions in the next sub-section.

---
### 6.2.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the GAN Accuracy and optimise the Computation Cost for training each GAN Model. These Callbacks will be main used during the GAN Training in the next sub-section. The revelant mathematical formula for the callbacks and logic are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 10 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 25 epochs, stop training and restore best weights}
$$

---

We will proceed to Define the Callbacks in the Code Cell below in preparation for the GAN Model Training.

In [ ]:
# ========== Define Learning Rate Scheduler (unchanged) =========#
lr_scheduler = LearningRateScheduler(
    lambda epoch, lr: lr * 0.95 if epoch % 10 == 0 else lr,
    verbose=1
)

# ========== Define Reduce LR on Plateau (unchanged) =========#
reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-6
)

# ========== Define Early Stopping (unchanged) =========#
early_stop = EarlyStopping(
    monitor='loss',
    patience=25,
    verbose=1,
    restore_best_weights=False
)

# ========== Define Display Generated Images for cDRAGAN =========#
def display_generated_images_c(generator, latent_dim, num_classes, n=7):

    # total = num_classes * n images
    total = num_classes * n
    noise = tf.random.normal([total, latent_dim])
    # create labels: 0 repeated n times, 1 repeated n times, ..., num_classes-1
    labels = tf.eye(num_classes)                       # shape (num_classes, num_classes)
    labels = tf.repeat(labels, repeats=n, axis=0)      # shape (num_classes*n, num_classes)

    gen_imgs = generator([noise, labels], training=False)
    gen_imgs = (gen_imgs + 1.0) / 2.0

    # plot as grid: rows = num_classes, cols = n
    fig, axes = plt.subplots(num_classes, n, figsize=(n, num_classes))
    for idx in range(total):
        row = idx // n
        col = idx % n
        img = gen_imgs[idx, :, :, 0]
        ax = axes[row, col] if num_classes > 1 else axes[col]
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# ========== Define FID Calculator (unchanged) =========#
def calculate_fid(real_images, fake_images, batch_size=10):
    # ... same as before ...
    real = (real_images + 1.0) * 127.5
    fake = (fake_images + 1.0) * 127.5
    real = tf.image.resize(real, (299,299))
    fake = tf.image.resize(fake, (299,299))
    if real.shape[-1] == 1:
        real = tf.image.grayscale_to_rgb(real)
        fake = tf.image.grayscale_to_rgb(fake)
    real_pp = preprocess_input(real)
    fake_pp = preprocess_input(fake)
    def _get_acts(x):
        acts = []
        for i in range(0, x.shape[0], batch_size):
            chunk = x[i:i+batch_size]
            acts.append(_inception_model(chunk, training=False).numpy())
        return np.vstack(acts)
    act_real = _get_acts(real_pp)
    act_fake = _get_acts(fake_pp)
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)
    diff = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(np.round(fid,4))

# ========== Confirmation Message =========#
print("cDRAGAN callbacks, scheduler, and visualisation setup complete.")

With reference to the Code Cell above, we are able to verify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a High Accuracy and Computing Efficient GAN Model Training.

---
### 6.2.3 Defining DRAGAN Generator Architecture

In this section, we will be defining the DRAGAN's Generator Architecture. The Generator will be trying to make realistic generated images in the attempt to bypass the Discriminator. Subsequently, the generated images from the Generator will be evaluated based on a few evaluation metrics such as Inception Score. The relevent mathematical formulas for the Loss Function of the Vanilla GAN Generator is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
With the relevant Generator Loss Formula indicated above, we will proceed to define the GAN Generator's Architecture in the Code Cell below and the Lost Function.

In [ ]:
# ========== Conditional Generator Function for cDRAGAN ==========
def build_generator(latent_dim=100, num_classes=16, use_dropout=True):
    # 1) Inputs
    noise_input = Input(shape=(latent_dim,), name="gen_noise")
    label_input = Input(shape=(num_classes,), name="gen_label")

    # 2) Concatenate
    x = Concatenate(name="gen_concat")([noise_input, label_input])

    # 3) Project & reshape (wider: 512 → 256 channels)
    x = Dense(7 * 7 * 512, use_bias=False, name="gen_dense")(x)
    x = BatchNormalization(name="gen_bn0")(x)
    x = Activation("relu", name="gen_act0")(x)
    x = Reshape((7, 7, 512), name="gen_reshape")(x)

    # 4) Upsample block 1: 7×7 → 7×7
    x = Conv2DTranspose(256, 5, strides=1, padding="same", use_bias=False, name="gen_deconv1")(x)
    x = BatchNormalization(name="gen_bn1")(x)
    x = Activation("relu", name="gen_act1")(x)
    if use_dropout:
        x = Dropout(0.1, name="gen_dropout1")(x)

    # 5) Upsample block 2: 7×7 → 14×14
    x = Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False, name="gen_deconv2")(x)
    x = BatchNormalization(name="gen_bn2")(x)
    x = Activation("relu", name="gen_act2")(x)
    if use_dropout:
        x = Dropout(0.1, name="gen_dropout2")(x)

    # 6) **New** Upsample block 3 (extra capacity): 14×14 → 14×14
    x = Conv2DTranspose(64, 5, strides=1, padding="same", use_bias=False, name="gen_deconv3")(x)
    x = BatchNormalization(name="gen_bn3")(x)
    x = Activation("relu", name="gen_act3")(x)
    if use_dropout:
        x = Dropout(0.1, name="gen_dropout3")(x)

    # 7) Final output layer: 14×14 → 28×28
    img_output = Conv2DTranspose(
        1, 5, strides=2, padding="same", activation="tanh", name="gen_output"
    )(x)

    return Model([noise_input, label_input], img_output, name="cDRAGAN_G")

# ========== cDRAGAN Generator Loss (unchanged) ==========
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_logits):
    return bce(tf.ones_like(fake_logits), fake_logits)

With reference to the Code Cell above, we are able to verify that the GAN's Generator Architecture have been successfully defined and we are able to proceed to define the GAN's Discriminator Architecture.

---
### 6.2.4 Defining DRAGAN Discriminator Architecture

In this section, we will be defining the Architecture of the Discriminator of the GAN. The Discriminator will be the Neural Network that will attempt to tell the Generated Images apart from the Actual Images. This will allow the GAN to improve overtime and also improve itself overtime. The mathematical function for the Discriminator Loss Function is indicated below:

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  
---

With the mathematical formula for the Loss Function indicated above, we will proceed to define the GAN's Discriminator Architecture and the Loss Function in the Code Cell below.

In [ ]:
# ========== Conditional Discriminator for cDRAGAN ==========
def build_discriminator(input_shape=(28, 28, 1), num_classes=16):
    # 1) Inputs
    img_input   = Input(shape=input_shape,               name="disc_image")
    label_input = Input(shape=(num_classes,),            name="disc_label")

    # 2) Embed label to a spatial map and concat with image
    x_lbl = Dense(input_shape[0] * input_shape[1],       name="disc_label_dense")(label_input)
    x_lbl = Reshape((input_shape[0], input_shape[1], 1), name="disc_label_reshape")(x_lbl)
    x     = Concatenate(axis=-1,                         name="disc_concat")([img_input, x_lbl])

    # 3) Conv block 1
    x = Conv2D(64, 5, strides=2, padding="same",         name="disc_conv1")(x)
    x = LeakyReLU(0.2,                                    name="disc_lrelu1")(x)
    x = Dropout(0.3,                                      name="disc_dropout1")(x)

    # 4) Conv block 2
    x = Conv2D(128, 5, strides=2, padding="same",        name="disc_conv2")(x)
    x = LeakyReLU(0.2,                                    name="disc_lrelu2")(x)
    x = Dropout(0.3,                                      name="disc_dropout2")(x)

    # 5) Final classification
    x = Flatten(name="disc_flatten")(x)
    logit = Dense(1, name="disc_output")(x)

    return Model([img_input, label_input], logit, name="cDRAGAN_D")

# ========== Discriminator Loss Function (with optional smoothing) ==========
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_logits, fake_logits, label_smoothing=0.9):
    real_targets = tf.ones_like(real_logits) * label_smoothing
    fake_targets = tf.zeros_like(fake_logits)

    real_loss = bce(real_targets, real_logits)
    fake_loss = bce(fake_targets, fake_logits)
    return real_loss + fake_loss

With reference to the Code Cell above, we are able to verify that the GAN's Discriminator Architecture have been successfully defined and we are able to proceed to define the GAN's Training Steps.

---
### 6.2.5 Defining Training Step

In this section, we will be Defining the Training Step function so as to train both the Generator and the Discriminator at the same time. The Training Step will compute the gradients for both the Generator and the Discriminator at the same time. The relevant mathematical formulas for the Training Step is indicated below:

---
**Generator Loss:**

Purpose: Produce Outputs Classified as Real.

$$
\mathcal{L}_G = \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - c)^2]
$$

Where:
- $c = 1$ = Desired Discriminator Output for Generated Images  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $z \sim p_z$ = Sample from Latent Space

---
**Discriminator Loss:**

Purpose: Discriminator Minimises the Least Squares Error between Predictions and True Labels.

$$
\mathcal{L}_D = \frac{1}{2} \mathbb{E}_{x \sim p_{\text{data}}}[(D(x) - b)^2] + \frac{1}{2} \mathbb{E}_{z \sim p_z}[(D(G(z)) - a)^2]
$$

Where:  
- $D(x)$ = Discriminator's Output for Real Data  
- $D(G(z))$ = Discriminator's Output for Generated Data  
- $a = 0$ = Label for Fake Data  
- $b = 1$ = Label for Real Data  
- $x \sim p_{\text{data}}$ = Sample from Real Data Distribution  
- $z \sim p_z$ = Sample from Latent Noise Distribution  

---
**Minimax Objective with DRAGAN Penalty:**

Purpose: LSGAN Seeks to Solve the Objective.

$$
\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{data}} [\log D(x)] + \mathbb{E}_{z \sim p_z(z)} [\log(1 - D(G(z)))] - \lambda \cdot \mathbb{E}_{\hat{x} \sim p_{perturbed}} \left[ \left( \|\nabla_{\hat{x}} D(\hat{x}) \|_2 - 1 \right)^2 \right]
$$
Where:
- $V(D, G)$ = Value Function for Adversarial Training
---

With the relevant mathematical formulas indicated above, we will proceed to define the Training Step in the Code Cell below.

In [ ]:
@tf.function
def train_step(
    real_images,
    real_labels,
    generator,
    discriminator,
    gen_optimizer,
    disc_optimizer,
    latent_dim,
    num_classes,
    gp_weight=1.0        # reduced gradient‐penalty weight
):
    batch_size = tf.shape(real_images)[0]

    # 1) One-hot encode real labels
    real_labels_oh = tf.one_hot(real_labels, depth=num_classes)  # shape (B, num_classes)

    # 2) Sample noise (no L2‐normalisation)
    noise = tf.random.normal([batch_size, latent_dim])          # shape (B, latent_dim)

    # 3) Sample random labels for generator
    rand_ints   = tf.random.uniform([batch_size], 0, num_classes, dtype=tf.int32)
    fake_labels = tf.one_hot(rand_ints, num_classes)            # shape (B, num_classes)

    # 4) No instance noise (use real_images directly)
    real_noisy = real_images

    with tf.GradientTape(persistent=True) as tape:
        # 5) Generate fake images conditioned on fake_labels
        fake_images = generator([noise, fake_labels], training=True)

        # 6) Discriminator outputs
        real_logits = discriminator([real_noisy, real_labels_oh], training=True)
        fake_logits = discriminator([fake_images, fake_labels], training=True)

        # 7) Discriminator loss (no label smoothing)
        d_loss_real = bce(tf.ones_like(real_logits), real_logits)
        d_loss_fake = bce(tf.zeros_like(fake_logits), fake_logits)
        d_loss = d_loss_real + d_loss_fake

        # 8) Generator loss
        g_loss = bce(tf.ones_like(fake_logits), fake_logits)

        # 9) DRAGAN gradient penalty (local perturbations)
        alpha         = tf.random.uniform([batch_size,1,1,1], 0.0, 1.0)
        noise_perturb = 0.5 * tf.random.normal(tf.shape(real_images))
        x_hat         = real_images + alpha * noise_perturb
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(x_hat)
            pred_hat = discriminator([x_hat, real_labels_oh], training=True)
        grads      = gp_tape.gradient(pred_hat, x_hat)
        grads      = tf.reshape(grads, [batch_size, -1])
        grads_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
        gp         = tf.reduce_mean((grads_norm - 1.0)**2)

        # 10) Add gradient penalty
        d_loss += gp_weight * gp

    # 11) Backpropagate
    d_grads = tape.gradient(d_loss, discriminator.trainable_variables)
    g_grads = tape.gradient(g_loss,     generator.trainable_variables)
    disc_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    gen_optimizer .apply_gradients(zip(g_grads, generator.trainable_variables))
    del tape

    return g_loss, d_loss, real_logits, fake_logits

With reference to the Code Cell above, we are able to determine that the Training Step have been successfully defined and we are able to proceed to prepare the Training Loop in the next section.

---
### 6.2.6 Defining Training Loop

In this section, we will be defining the Training Loop for the GAN Training. The Training Loop will go through each of the Epoch and train both the Generator and Discriminator simultaneously. The Training Loop will also track the Loss for both the Generator and the Discriminator for each of the Epoch trained. The relevant mathematical formulas are indicated below:

---
**Epoch Loss Averaging:**

Purpose: These are the Average Generator and Discriminator Losses over all Batches in Epoch $t$.

$$
\bar{L}_G^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_G^{(i)} \\
\bar{L}_D^{(t)} = \frac{1}{N} \sum_{i=1}^{N} L_D^{(i)}
$$

Where:
- $\bar{L}_G^{(t)}$ = Average Generator Loss at Epoch $t$  
- $\bar{L}_D^{(t)}$ = Average Discriminator Loss at Epoch $t$  
- $N$ = Number of Batches in the Dataset  
- $L_G^{(i)}$ = Generator Loss on Batch $i$  
- $L_D^{(i)}$ = Discriminator Loss on Batch $i$

---
**Epoch Iteration:**

Purpose: Describes the Training Loop Logic: for Each Epoch $t$, perform `train_step` for Every Mini-batch.

$$
\text{for } t = 1 \text{ to } T:
\quad \text{for each batch } (x^{(i)}):
\quad \text{train_step}(x^{(i)})
$$

Where:
- $T$: Total Number of Epochs  
- $x^{(i)}$: Real Batch $i$ from the Dataset  
- `train_step`: Function that Updates $G$ and $D$ using that Batch

---
**Generated Image Output:**

Purpose: Sample Random Noise $z$ and Generate Synthetic Image $\hat{x}$ from the Generator. This is used for Visual Monitoring of Model Quality.

$$
z \sim p_z(z) \\
\hat{x} = G(z)
$$

Where:
- $z$: Random Latent Vector  
- $G(z)$: Generated Image from Generator

---
With the mathematical formulas indicated above, we will proceed to define the Training Loop in the Code Cell below in preparation of the GAN Training.

In [ ]:
def train_cDRAGAN(
    dataset, epochs, generator, discriminator,
    gen_optimizer, disc_optimizer,
    latent_dim, num_classes,
    save_path=None, use_early_stopping=True,
    steps_per_epoch=None, gp_weight=2.0
):
    # ----- Prepare Fixed Real Batch from Validation Set ----- #
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(50).take(1)
    real_val_imgs, real_val_lbls = next(iter(val_ds))
    real_val_lbls_oh = tf.one_hot(real_val_lbls, depth=num_classes)

    # ----- Prepare Fixed Noise Batch for FID ----- #
    noise_val = tf.random.normal([50, latent_dim])

    # ----- Early Stopping & Checkpoint Variables ----- #
    best_fid     = float('inf')
    no_improve   = 0
    patience     = 10
    fid_interval = 1

    # ----- Dummy Model for Keras Callbacks ----- #
    dummy_in  = Input(shape=(1,))
    dummy_out = Lambda(lambda x: x)(dummy_in)
    dummy     = Model(dummy_in, dummy_out)
    dummy.compile(optimizer=Adam(1e-4), loss='mse')

    early_stop.set_model(dummy)
    reduce_lr .set_model(dummy)
    lr_scheduler.set_model(dummy)
    early_stop.on_train_begin({})
    reduce_lr .on_train_begin({})
    lr_scheduler.on_train_begin({})

    gen_loss_history, disc_loss_history = [], []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        epoch_g_losses, epoch_d_losses = [], []

        # ----- Per-batch Training ----- #
        for i, (real_images, real_labels) in enumerate(dataset):
            if steps_per_epoch and i >= steps_per_epoch:
                break

            g_loss, d_loss, _, _ = train_step(
                real_images, real_labels,
                generator, discriminator,
                gen_optimizer, disc_optimizer,
                latent_dim, num_classes,
                gp_weight=gp_weight
            )
            epoch_g_losses.append(g_loss)
            epoch_d_losses.append(d_loss)

        # ----- Log Epoch Losses ----- #
        avg_g = tf.reduce_mean(epoch_g_losses)
        avg_d = tf.reduce_mean(epoch_d_losses)
        print(f"Generator Loss: {avg_g:.4f} | Discriminator Loss: {avg_d:.4f}")
        gen_loss_history.append(float(avg_g))
        disc_loss_history.append(float(avg_d))

        # ----- Callbacks on Generator Loss ----- #
        logs = {'loss': float(avg_g)}
        if use_early_stopping:
            early_stop.on_epoch_end(epoch, logs)
            if early_stop.stopped_epoch > 0:
                print(f"Early stopping at epoch {epoch}")
                break
        reduce_lr.on_epoch_end(epoch, logs)
        lr_scheduler.on_epoch_end(epoch, logs)

        # ----- FID Evaluation & Checkpointing ----- #
        if epoch % fid_interval == 0:
            fake_val = generator([noise_val, real_val_lbls_oh], training=False)
            real_fid = tf.image.resize(real_val_imgs, [299, 299])
            fake_fid = tf.image.resize(fake_val,    [299, 299])
            fid_value = calculate_fid(real_fid, fake_fid)

            print(f"Epoch {epoch} → Val FID: {fid_value:.2f}")
            if fid_value < best_fid:
                best_fid   = fid_value
                no_improve = 0
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    generator.save_weights(os.path.join(save_path, "best_gen.weights.h5"))
                    discriminator.save_weights(os.path.join(save_path, "best_disc.weights.h5"))
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"No FID improvement for {patience} epochs, stopping.")
                break

        # ----- Display Conditional Samples Every 5 Epochs ----- #
        if epoch % 5 == 0:
            display_generated_images_c(generator, latent_dim, num_classes)

    # ----- Restore Best Weights After Training ----- #
    if save_path:
        best_g = os.path.join(save_path, "best_gen.weights.h5")
        best_d = os.path.join(save_path, "best_disc.weights.h5")
        if os.path.exists(best_g) and os.path.exists(best_d):
            generator.load_weights(best_g)
            discriminator.load_weights(best_d)
            print("Restored best weights based on lowest FID.")
        else:
            print("Best weights not found; skipping restore.")

    return float(avg_g), gen_loss_history, disc_loss_history

With reference to the Code Cell above, we are able to determine that the Training Loop have been successfully defined and we are able to proceed to define the Optimisers with Tunable Parameter in preparation for the Optuna Tuning.

---
### 6.2.7 Defining Optimisers with Tunable Parameters

In this section, we will be defining the Common Optimisers that will be Tuned using the Optuna Tuning. The Optimisers will each have a range of values in a list so as to allow for the Optuna to search for the best Hyperparameter for the GAN. Through these parameters, it will affect how effective and efficiently the GAN will learn from the provided EMNIST Data. The relevent mathematical formula is indicated below:

---
**Optimiser Update Rule**

Purpose: Parameter Update Step in the Adam Optimiser, using Adaptive Learning Rates with Momentum.

$$
\theta \leftarrow \theta - \eta \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}
$$

Where:
- $\theta$ = Model Parameters (Generator or Discriminator)
- $\eta$ = Learning Rate (Tuned)
- $m_t$ = First Moment Estimate (Mean of Gradients)
- $v_t$ = Second Moment Estimate (Variance of Gradients)
- $\epsilon$ = Small Constant for Numerical Stability

---
With the mathematical formula indicated above, we will proceed to define the Optimisers in the Code Cell.

In [ ]:
# ========== Conditional Objective Function for cDRAGAN ========== #
def objective(trial):
    # ----- Hyperparameters ----- #
    latent_dim    = trial.suggest_categorical('latent_dim', [100, 128, 160])
    learning_rate = trial.suggest_float('learning_rate', 5e-5, 2e-4, log=True)
    beta_1        = trial.suggest_float('beta_1', 0.4, 0.6)

    # ----- Build Conditional Models ----- #
    generator     = build_generator(latent_dim=latent_dim, num_classes=16)
    discriminator = build_discriminator(num_classes=16)

    # ----- Trigger Weight Creation ----- #
    _ = generator([tf.random.normal([1, latent_dim]), tf.one_hot([0], 16)])
    _ = discriminator([tf.random.normal([1, 28, 28, 1]), tf.one_hot([0], 16)])

    # ----- Define Optimizers ----- #
    gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

    # ----- Prebuild Optimizer Variables ----- #
    _ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
    _ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

    # ----- Load Conditional Training Data ----- #
    dataset = load_data(X_train, y_train, batch_size=64)
    steps   = min(1000, len(X_train) // 64)

    # ----- Train cDRAGAN ----- #
    _ = train_cDRAGAN(
        dataset=dataset,
        epochs=50,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        num_classes=16,
        steps_per_epoch=steps,
        save_path=None,
        gp_weight=5.0
    )

    # ----- Generate Fake Images for FID ----- #
    z = tf.random.normal([1000, latent_dim])
    rand_ints   = tf.random.uniform([1000], 0, 16, dtype=tf.int32)
    fake_labels = tf.one_hot(rand_ints, 16)
    fake_images = generator([z, fake_labels], training=False)

    # ----- Real Images from Validation Set ----- #
    real_images = X_val[:1000]

    # ----- Compute FID Score ----- #
    fid_score = calculate_fid(real_images, fake_images)
    return fid_score

With reference to the Code Cell above, we are able to determine that the Optimiser have been successfully defined with a range of different Parameter Values so as to provide a robust Optuna Tuning in the subsequent sections.

---
### 6.2.8 Optuna Tuning Study

In this sub-section, we will be conducting the Optuna Tuning Study on the GAN Model so as to obtain the best Hyperparameter for the GAN Model. It is expected that the Tuning takes a prolonged duration due to the nature of how the GAN works and the 2 Neural Networks involved. However, the training data will be stored so as to prevent re-running of repetitive codes. The Optuna Tuning Study will be conducted in the Code Cell below.

In [ ]:
# ========== Create or Load Study with SQLite Backend for cDRAGAN =========#
study = optuna.create_study(
    direction="minimize",
    study_name="cdragan_tuning",
    storage="sqlite:////content/drive/MyDrive/Colab Notebooks/DELE CA2 A/Non-Augmented GAN Tunings/cdragan_optuna.db",
    load_if_exists=True
)

# ========== Trial Management =========#
MAX_TRIALS = 50
completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining_trials = MAX_TRIALS - completed_trials

if remaining_trials > 0:
    print(f"Resuming cDRAGAN study: {completed_trials} completed, running {remaining_trials} more.")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"cDRAGAN study already completed {MAX_TRIALS} trials. Skipping optimization.")

# ========== Output Best Trial =========#
best_trial     = study.best_trial
best_params_df = pd.DataFrame([best_trial.params])
best_params_df["Final FID"] = best_trial.value
best_params_df.style.background_gradient(cmap="Blues")

With reference to the Code Cell above, we are able to view the Best Hyperparameter obtained during the Optuna Tuning Study. This Hyperparameter will be extracted and be trained for a longer period of time so as to attempt to increase the performance and limit the loss for the GAN.

---
### 6.2.9 GAN Re-Training Operation

As obtained from the previous sub-section, we will be re-training the GAN using the best Hyperparameter obtained during the Optuna Tuning Study. This is to push the GAN Model to its limits and also to save Computational Power as we are not taking up prolonged periods of time tuning GAN Models which may not have any clear signs of good performance. The retraining of the GAN Model will be conducted in the Code Cell below.

In [ ]:
# ========== Extract Best Parameters from Study ========== #
best_params = study.best_trial.params

latent_dim    = best_params['latent_dim']
learning_rate = best_params['learning_rate']
beta_1        = best_params['beta_1']

print("Using Best Trial Parameters for cDRAGAN:")
print(best_params)

# ========== Rebuild cDRAGAN Models ========== #
generator     = build_generator(latent_dim=latent_dim, num_classes=16)
discriminator = build_discriminator(num_classes=16)

# ========== Initialise Optimisers ========== #
gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=beta_1)

# ========== Pre‐build Optimiser Variables ========== #
_ = gen_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in generator.trainable_variables])
_ = disc_optimizer.apply_gradients([(tf.zeros_like(v), v) for v in discriminator.trainable_variables])

# ========== Reload Conditional Dataset ========== #
dataset = load_data(X_train, y_train, batch_size=64)

# ========== Define Paths ========== #
model_name = "cdragan"
output_dir = f"/content/drive/MyDrive/Colab Notebooks/DELE CA2 A/final_outputs_{model_name}"
weights_path = os.path.join(output_dir, f"{model_name}_generator_final.weights.h5")
gen_ckpt     = os.path.join(output_dir, "best_gen.weights.h5")
disc_ckpt    = os.path.join(output_dir, "best_disc.weights.h5")
loss_csv     = os.path.join(output_dir, "loss_history.csv")

os.makedirs(output_dir, exist_ok=True)

# ========== Skip if Already Trained ========== #
if os.path.exists(gen_ckpt) and os.path.exists(disc_ckpt) and os.path.exists(loss_csv):
    print("All outputs found. Skipping training...")

    generator.load_weights(gen_ckpt)
    discriminator.load_weights(disc_ckpt)
    print("Generator and Discriminator Weights Loaded.")

    loss_df = pd.read_csv(loss_csv)
    gen_loss_history  = loss_df['gen_loss'].tolist()
    disc_loss_history = loss_df['disc_loss'].tolist()
    final_gen_loss    = gen_loss_history[-1]

else:
    # ========== Train cDRAGAN ========== #
    final_gen_loss, gen_loss_history, disc_loss_history = train_cDRAGAN(
        dataset=dataset,
        epochs=100,
        generator=generator,
        discriminator=discriminator,
        gen_optimizer=gen_optimizer,
        disc_optimizer=disc_optimizer,
        latent_dim=latent_dim,
        num_classes=16,
        use_early_stopping=False,
        steps_per_epoch=1546,
        save_path=output_dir,
        gp_weight=10.0
    )

    # ========== Save Final Generator Weights ========== #
    generator.save_weights(weights_path)
    print("Generator Weights Saved.")

    # ========== Save Loss History ========== #
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(gen_loss_history) + 1)),
        'gen_loss': gen_loss_history,
        'disc_loss': disc_loss_history
    })
    loss_df.to_csv(loss_csv, index=False)
    print(f"Loss history saved to {loss_csv}")

With reference to the Code Cell above, we are able to determine that the best GAN Model have been trained and saved successfully and we are able to proceed with the Model Evaluation in the next sub-sections where we determine how well the GAN Model performed based on various metrics and evaluation methodologies.

---
### 6.2.10 GAN Model Evaluation

In this sub-section, we will be evaluating the GAN's Generator performance on various aspects. We will be utilising different methodologies and metrics to attempt to quantify the performance of the GAN Generator while also conducting Visual Inspection on the result output of the GAN. The Evaluation will be conducted in the following sections.

---
#### 6.2.10.1 Visual Grid of Samples

In this sub-section, we will be visualising the Grid of Samples of the Synthesised Data generated by the GAN Model. A list of observations will be made before the utilisation of Quantitative Metrics to evaluate the GAN Generator's general performance. As the observations made are purely from professional opinion, it will most likely not be used as a Basis of Comparison between GAN Models due to differing opinions unless in extreme cases. The visualisation will be conducted in the Code Cell below.

In [ ]:
def plot_generated_images_grid(generator, latent_dim, num_classes, images_per_class=10, save_path=None, title="cDRAGAN Generated Images", seed=42):
    total = num_classes * images_per_class

    # ----- Seed for Reproducibility ----- #
    if seed is not None:
        tf.random.set_seed(seed)

    # ----- Sample Noise and Labels ----- #
    noise = tf.random.normal([total, latent_dim])
    class_indices = tf.repeat(tf.range(num_classes), images_per_class)
    labels = tf.one_hot(class_indices, depth=num_classes)

    # ----- Generate and Rescale from [-1,1] to [0,1] ----- #
    gen_imgs = generator([noise, labels], training=False)
    gen_imgs = (gen_imgs + 1.0) / 2.0
    gen_imgs = tf.clip_by_value(gen_imgs, 0.0, 1.0).numpy()

    # ----- Plot Grid without Labels ----- #
    fig, axes = plt.subplots(
        images_per_class, num_classes,
        figsize=(num_classes, images_per_class),
        squeeze=False
    )
    for idx in range(total):
        row = idx % images_per_class
        col = idx // images_per_class
        axes[row, col].imshow(gen_imgs[idx, :, :, 0], cmap='gray')
        axes[row, col].axis('off')

    plt.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved image grid to: {save_path}")
    plt.show()


# ========== Example Call ==========
plot_generated_images_grid(generator, latent_dim=160, num_classes=16, images_per_class=10)

With reference to the output of the Visual Grid of Samples above, we are able to make the following observations:

**Basic Structure of Letters is Captured**

- Many characters resemble actual letters such as "A", "P", "D", "F", "L", "O", "X", and "Z".

- Most generated samples exhibit rounded strokes and centring typical of EMNIST letters.

**Blur and Stroke Distortion**

- Several characters are over-smoothed or excessively noisy, resulting in illegible or smeared outputs.

- Characters like "e", "g", and "r" appear melty or fragmented, suggesting that mode stability is lacking.

**Character Consistency**

- Across identical row positions (which ideally should produce similar classes), there is significant variation in stroke quality and formation.

- For instance, characters supposed to represent "B", "F", or "J" have wildly inconsistent forms.

With these visual observations completed, we will now proceed to visualise the Loss Curve in the next sub-section.

---
#### 6.2.10.2 Loss Curve Over Epochs

In this sub-section, we will be plotting the GAN Loss Curve over Epochs to evaluate the training behaviour of the GAN. This curve provides insight into whether the GAN is experiencing underfitting, overfitting, or achieving a stable training dynamic between the generator and discriminator.

Although GANs do not minimise a single unified loss in the traditional sense, the trajectory of the generator and discriminator losses can indicate whether the adversarial training process is converging appropriately.

If the generator loss remains high while discriminator loss quickly drops to near-zero, this may indicate **underfitting**, where the generator is not learning effectively to produce plausible images.

If the discriminator loss increases while generator loss sharply decreases, this may suggest **overfitting** or **mode collapse**, where the generator exploits narrow weaknesses in the discriminator without general improvement.

A relatively stable oscillation or convergence between both losses typically signifies a **well-balanced** training process, where the generator and discriminator are learning in tandem.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Loss Curve**

- Increase training epochs to allow generator more time to learn.

- Simplify the discriminator to reduce overpowering the generator.

- Adjust the learning rate or apply label smoothing.

- Consider adding batch normalisation in the generator.

**Overfitting or Mode Collapse Loss Curve**

- Introduce or increase dropout in the discriminator.

- Apply input noise or label flipping to the discriminator.

- Evaluate diversity of generated samples regularly.

**Balanced/Ideal Loss Curve**

- Maintain current architecture and hyperparameters.

- Proceed with full-scale image generation.

- Evaluate both qualitative (visual inspection) and quantitative metrics (e.g. Inception Score or FID).

With these potential training behaviours and action plan outlined, we will proceed to visualise the GAN loss dynamics in the code cell below.

In [ ]:
# =========== Figure Size Configuration =========== #
plt.figure(figsize=(10, 6))

# =========== Plot Training Losses =========== #
plt.plot(gen_loss_history, label="Generator Loss", linewidth=2, color='blue')
plt.plot(disc_loss_history, label="Discriminator Loss", linewidth=2, color='red')

# =========== Labels and Title =========== #
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("DRAGAN Training Loss Curve (Non-Augmented)", fontsize=16)

# =========== Grid, Legend, and Styling =========== #
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# =========== Plot Display =========== #
plt.tight_layout()
plt.show()

With reference to the output of the Loss Curve above, we are able to make the following observations:

- Stability is achieved, but diversity and realism in generated images remain limited, as reflected in the earlier visual grid.

- DRAGAN appears to be resistant to collapse, but it may require architectural improvements or tuning to further reduce discriminator overconfidence and stimulate better generator learning.

With the observations indicated, we will proceed to conduct the Quantitave Metrics Evaluation in the next sub-section.

---
#### 6.2.10.3 Quantitative Metrics Evaluation

In this sub-section we will be utilising Quantitative Metrics such as FID and KID to tabulate the results of the performance and quantify the performance of the GAN Generator. We will subsequently be use these metrics to conduct inter-model evaluation after tuning and training all the GANs. The metrics that we will be utilising and their respective formulas are indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
def evaluate_gan_model(generator, latent_dim, num_classes, X_val, gen_loss_history, disc_loss_history, model_name="cDRAGAN"):
    # ----- Generate 1000 Fake Images with Random Classes ----- #
    n_samples = 1000
    noise = tf.random.normal([n_samples, latent_dim])
    rnd_classes = tf.random.uniform([n_samples], 0, num_classes, dtype=tf.int32)
    fake_labels = tf.one_hot(rnd_classes, depth=num_classes)
    fake_images = generator([noise, fake_labels], training=False)

    # ----- Prepare Real Images (First 1000) in [-1,1] ----- #
    real_images = tf.convert_to_tensor(X_val[:n_samples], dtype=tf.float32)

    # ----- Compute FID & KID ----- #
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)

    # ----- Compute t-SNE Spread ----- #
    fake_np = ((fake_images + 1.0) / 2.0).numpy().reshape(n_samples, -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(fake_np)
    tsne_spread = (proj[:,0].max() - proj[:,0].min()) * (proj[:,1].max() - proj[:,1].min())
    tsne_spread = round(tsne_spread, 2)

    # ----- Assess Mode Collapse ----- #
    unique = np.unique(np.round(fake_np,3), axis=0).shape[0]
    collapse_risk = "Low" if unique > 0.9 * n_samples else "High"

    # ----- Compute PPL ----- #
    distances = []
    eps = 1e-2
    for _ in range(50):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + eps * tf.random.normal([1, latent_dim])
        img1 = generator([z1, tf.one_hot([0], num_classes)], training=False)
        img2 = generator([z2, tf.one_hot([0], num_classes)], training=False)
        p1 = tf.image.resize(tf.image.grayscale_to_rgb((img1+1)*127.5), (128,128))
        p2 = tf.image.resize(tf.image.grayscale_to_rgb((img2+1)*127.5), (128,128))
        f1 = vgg_model(vgg_preprocess(p1))
        f2 = vgg_model(vgg_preprocess(p2))
        distances.append(tf.reduce_mean(tf.square(f1-f2)).numpy())
    ppl_value = round(np.mean(distances), 4)

    # ----- Loss Stats ----- #
    mean_g, std_g = round(np.mean(gen_loss_history),4), round(np.std(gen_loss_history),4)
    mean_d, std_d = round(np.mean(disc_loss_history),4), round(np.std(disc_loss_history),4)

    # ----- Assemble DataFrame ----- #
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_g,
        "Std Generator Loss": std_g,
        "Mean Discriminator Loss": mean_d,
        "Std Discriminator Loss": std_d,
        "Mode Collapse Risk": collapse_risk,
        "Diversity (t-SNE Spread)": tsne_spread,
        "PPL": ppl_value
    }
    df = pd.DataFrame([row])
    return df[[
        "Model","FID Score","KID Score",
        "Mean Generator Loss","Std Generator Loss",
        "Mean Discriminator Loss","Std Discriminator Loss",
        "Mode Collapse Risk","Diversity (t-SNE Spread)","PPL"
    ]]

# ========== Create DataFrame ========== #
df = evaluate_gan_model(generator, latent_dim=160, num_classes=16, X_val=X_val, gen_loss_history=gen_loss_history, disc_loss_history=disc_loss_history, model_name="cDRAGAN")

# ========== Display DataFrame ========== #
df.style.background_gradient(cmap="Blues")

With reference to the Quantitative Metrics above, we are able to determine the the Conditonal-DRAGAN is a better model as it surpass the other GAN Models in both the FID and KID Score. Therefore, the final GAN Model will be the cDRAGAN.

---
#### 6.2.10.4 t-SNE of Generated Samples

In this sub-section, we visualise the t-SNE projection of generated samples to evaluate the latent diversity learned by the unconditional GAN. Although our GAN is not class-conditioned, t-SNE remains a valuable tool to assess whether the generator is producing varied outputs that reflect meaningful use of the latent space. While GANs are trained in high-dimensional spaces, t-SNE enables us to observe 2D structural patterns such as sample clustering, separation, and density, which provide indirect insights into the diversity and generalisation capabilities of the generator. The visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Generate New Images from Random Noise ========== #
latent_dim   = 160
num_classes  = 16
n_samples    = 500

# ========== Sample Gaussian Noise ========== #
noise = tf.random.normal([n_samples, latent_dim])

# ========== Sample Random Class Labels and One‐hot Encode ========== #
rand_ints = tf.random.uniform([n_samples], 0, num_classes, dtype=tf.int32)
labels_oh = tf.one_hot(rand_ints, depth=num_classes)  # shape (n_samples, num_classes)

# ========== Generate Images ========== #
generated_images = generator([noise, labels_oh], training=False)


def plot_tsne_embeddings(
    generated_images,
    save_path=None,
    title="t-SNE of cDRAGAN Generated Samples"
):
    # ----- Check input range ----- #
    mi, ma = tf.reduce_min(generated_images).numpy(), tf.reduce_max(generated_images).numpy()
    if not (mi >= -1.0 and ma <= 1.0):
        raise ValueError(f"Images must be in [-1,1]. Got range [{mi:.3f},{ma:.3f}]")

    # ----- Rescale from [-1,1] to [0,1] ----- #
    images_rescaled = (generated_images + 1.0) / 2.0
    flat_images     = images_rescaled.numpy().reshape(n_samples, -1)

    # ----- Run t-SNE ----- #
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                init='pca', random_state=42)
    tsne_proj = tsne.fit_transform(flat_images)

    # ----- Plot t-SNE ----- #
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        x=tsne_proj[:, 0],
        y=tsne_proj[:, 1],
        s=20,
        alpha=0.9,
        edgecolor='none'
    )
    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"t-SNE plot saved to: {save_path}")

    plt.show()

# ========== Call Plot Function ========== #
plot_tsne_embeddings(generated_images)

With reference to the t-SNE of cDRAGAN Generated Samples above, we are able to make the following observations:

- There is a broad scattering of points across the t-SNE space, suggesting moderate diversity with less mode collapse than DRAGAN or LSGAN.

- However, the semantic alignment across samples is inconsistent, weakening the correlation between visual diversity and actual meaningful variation.

With the observations stated, we are able to conclude that the best GAN Model is the cDRAGAN.

---
## 6.3 GAN Comparison to VAE

In this section, we will be comparing the Best GAN Model (cDRAGAN) to a VAE so as to evaluate the performance of GAN Models as compared to VAEs as both are generative models. Through this comparison, we will be able to attempt to justify why a VAE or GAN may be more advantages in generative task for the EMNIST Dataset. The comparison will be conuducted in this section.

---
### 6.3.1 Defining Sampling Layer

In this sub-section, we will be defining the Sampling Layer for the VAE. The Sampling Layer will then subsequently be used in the Encoder's Architecture as the final layer. The relevant mathematical formulas are indicated below:

**Sampling Layer**

Purpose: Enable differentiable sampling from the encoder's Gaussian latent distribution via the reparameterization trick.

$$
\sigma = \exp\!\Bigl(\tfrac{1}{2}\,\log \sigma^2\Bigr)
$$

$$
z = \mu + \sigma \;\odot\; \epsilon,\quad \epsilon \sim \mathcal{N}(0, I)
$$


Where:
- $\mu$ (z_mean) = Learned Mean Vector of the Latent Distribution  
- $\sigma^2$ = Learned Variance Vector
- $\sigma$ = Standard Deviation Vector, $\sqrt{\sigma^2}$  
- $\epsilon$ = Noise Sampled from $\mathcal{N}(0, I)$  
- $\odot$ = Element-wise Multiplication  
- $z$ = Sampled Latent Vector Passed to Decoder  

With the relevant mathematical formulas indicated above, we will proceed to define the Sampling Layer in the Code Cell below.

In [ ]:
# ========== Sampling Layer ========== #
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

With reference to the Code Cell above, we are able to verify that the Sampling Layer have been successfully defined.

---
### 6.3.2 Defining Encoder Architecture

In this sub-section, we will be defining the Encoder Architecture so as to encode the input data before decoding the data. The Encoder will operate alongside the Decoder so as to generate synthesise new data. The relevant mathematical formulas are indicated below:

---

**First Convolutional Layer**

$$
h^{(1)} = \mathrm{ReLU}\bigl(\mathrm{Conv}_1(x)\bigr)
$$

Where:  
- $x$ = Input Image Tensor of Shape $(28,28,1)$  
- $\mathrm{Conv}_1$ = First Convolutional Layer
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  
- $h^{(1)}$ = Output Feature Map after First Convolution

---

**Second Convolutional Layer**

$$
h^{(2)} = \mathrm{ReLU}\bigl(\mathrm{Conv}_2(h^{(1)})\bigr)
$$

Where:
- $h^{(1)}$ = Input Feature Map from Previous Layer  
- $\mathrm{Conv}_2$ = Second Convolutional Layer
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  
- $h^{(2)}$ = Output Feature Map after Second Convolution

---

**Flatten Layer**

$$
h = \mathrm{Flatten}\bigl(h^{(2)}\bigr)
$$

Where:
- $h^{(2)}$ = Input Feature Map of Shape
- $\mathrm{Flatten}$ = Operation Converting a Tensor into a 1D Vector  
- $h$ = Flattened Vector Representation

---

**Dense Hidden Layer**

$$
h_{d} = \mathrm{ReLU}\bigl(W_{h}\,h + b_{h}\bigr)
$$

Where:
- $h$ = Flattened Input Vector  
- $W_{h}$ = Weight Matrix of the Dense Hidden Layer  
- $b_{h}$ = Bias Vector of Dense Hidden Layer  
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  
- $h_{d}$ = Activated Hidden Representation

---

**Latent Mean**

$$
\mu = W_{\mu}\,h_{d} + b_{\mu}
$$

Where:
- $h_{d}$: Hidden Representation Input  
- $W_{\mu}$: Weight Matrix Mapping to Latent Mean  
- $b_{\mu}$: Bias Vector for Latent Mean  
- $\mu$: Mean Vector of Gaussian Latent Distribution

---

**Latent Log-Variance**

$$
\log \sigma^2 = W_{\log \sigma^2}\,h_{d} + b_{\log \sigma^2}
$$

Where:
- $h_{d}$ = Hidden Representation Input  
- $W_{\log \sigma^2}$ = Weight Matrix Mapping to Log-variance  
- $b_{\log \sigma^2}$ = Bias Vector for Log-variance  
- $\log \sigma^2$ = Log-variance Vector of Gaussian Latent Distribution

With the relevant mathematical formulas indicated above, we will be proceeding to define the Encoder Architecture in the Code Cell below.

In [ ]:
# ========== Defining Encoder Architecture ========== #
latent_dim = 16

encoder_inputs = keras.Input(shape=(28,28,1))
x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])

# ========== Encoder Architecture Overview ========== #
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
encoder.summary()

With referene to the output of the Code Cell above, we are able to determine that the Encoder Architecture have been defined successfully and we are also able to see the overview of the Encoder's Architecture Layers based on the Summary above.

---
### 6.3.3 Defining Decoder Architecture

In this sub-section, we will be defining the Decoder Architecture which will complement the Encoder Architecture which was previously defined. The Decoder will the Decoding the Encoded Data. The relevant mathematical formulas are indicated below:

---

**Dense Expansion Layer**  
$$
h^{(1)} = \mathrm{ReLU}\bigl(W^{(1)}\,z + b^{(1)}\bigr)
$$

Where:
- $z$ = Latent Vector of Shape $(d,)$  
- $W^{(1)}, b^{(1)}$ = Weights and Biases of Dense Expansion Layer  
- $h^{(1)}$ = Expanded Dense Representation
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  

---

**Reshape to Feature Map**  
$$
h^{(2)} = \mathrm{Reshape}\bigl(h^{(1)},\,(7,7,64)\bigr)
$$

Where:  
- $h^{(1)}$ = Dense Vector Input  
- $\mathrm{Reshape}$ = Operation Converting Vector to Tensor of Shape
- $h^{(2)}$ = Feature Map Tensor of Shape  

---

**First Transposed Convolution**  
$$
h^{(3)} = \mathrm{ReLU}\bigl(\mathrm{ConvT}_1(h^{(2)})\bigr)
$$

Where:  
- $\mathrm{ConvT}_1$ = Conv2DTranspose Layer  
- $h^{(2)}$ = Input Feature Map  
- $h^{(3)}$ = Output Feature Map of Shape  
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  

---

**Second Transposed Convolution**
$$
h^{(4)} = \mathrm{ReLU}\bigl(\mathrm{ConvT}_2(h^{(3)})\bigr)
$$

Where:  
- $\mathrm{ConvT}_2$ = Conv2DTranspose Layer
- $h^{(3)}$ = Input Feature Map  
- $h^{(4)}$ = Output Feature Map of Shape  
- $\mathrm{ReLU}$ = Rectified Linear Unit Activation  

---

**Output Reconstruction Layer**  
$$
\hat{x} = \tanh\bigl(\mathrm{ConvT}_3(h^{(4)})\bigr)
$$

Where:  
- $\mathrm{ConvT}_3$ = Conv2DTranspose Layer
- $\tanh$ = Hyperbolic Tangent Activation Mapping Outputs to $[-1,1]$  
- $\hat{x}$ = Reconstructed Image Tensor of Shape

With the relevant mathematical formulas indicated above, we will proceeed to define the Decoder Architecture in the Code Cell below.

In [ ]:
# ========== Defining Decoder Architecture ========== #
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(7*7*64, activation='relu')(latent_inputs)
x = layers.Reshape((7,7,64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation='tanh', padding='same')(x)

# ========== Decoder Architecture Overview ========== #
decoder = keras.Model(latent_inputs, decoder_outputs, name='decoder')
decoder.summary()

With referene to the output of the Code Cell above, we are able to determine that the Decoder Architecture have been defined successfully and we are also able to see the overview of the Decoder's Architecture Layers based on the Summary above.

---
### 6.3.4 Defining VAE Model

In this sub-section, we will be defining the Variational Auto Encoder Model with the Encoder and Decoder combined. In the class, it will include the train step and also the Loss calculation. The relevant mathematical formulas are indicated below:

---

**Reconstruction Loss (MSE)**  
$$
\mathcal{L}_{\text{recon}}
= \frac{1}{N}\sum_{i=1}^{N}\sum_{p}\bigl(x_{i}^{(p)} - \hat{x}_{i}^{(p)}\bigr)^{2}
$$  
Where:  
- $N$ = Batch size  
- $i$ = Sample Index in Batch  
- $p$ = Pixel Index  
- $x_{i}^{(p)}$ = True Value of Pixel $p$ in Sample $i$  
- $\hat{x}_{i}^{(p)}$ = Reconstructed Value of Pixel $p$ in Sample $i$  

---

**KL Divergence Loss**  
$$
\mathcal{L}_{\text{KL}}
= -\frac{1}{2N}\sum_{i=1}^{N}\sum_{j=1}^{d}\Bigl(1 + \log\sigma_{i,j}^{2} - \mu_{i,j}^{2} - \sigma_{i,j}^{2}\Bigr)
$$  
Where:  
- $d$ = Dimensionality of the Latent Space  
- $j$ = Latent Dimension Index  
- $\mu_{i,j}$ = Mean of Latent Dimension $j$ for Sample $i$  
- $\log\sigma_{i,j}^{2}$ = Log-variance of Latent Dimension $j$ for Sample $i$  
- $\sigma_{i,j}^{2} = \exp\bigl(\log\sigma_{i,j}^{2}\bigr)$ = Variance  

---

**Total VAE Loss**  
$$
\mathcal{L}_{\text{total}}
= \mathcal{L}_{\text{recon}} \;+\; \mathcal{L}_{\text{KL}}
$$  
Where:  
- $\mathcal{L}_{\text{total}}$ = Combined Loss Optimized during Training  
- $\mathcal{L}_{\text{recon}}$ = Reconstruction Loss  
- $\mathcal{L}_{\text{KL}}$ = KL Divergence Loss

---

With the relevant mathematical formula indicated above, we will proceed with defining the VAE Model in the Code Cell below.

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.recon_loss_tracker = keras.metrics.Mean(name="recon_loss")
        self.kl_loss_tracker    = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.recon_loss_tracker,
            self.kl_loss_tracker
        ]

    def compute_losses(self, data):
        z_mean, z_log_var, z = self.encoder(data, training=False)
        reconstruction = self.decoder(z, training=False)
        # ----- Reconstruction Loss ----- #
        recon_loss = tf.reduce_mean(
            tf.reduce_sum(
                tf.square(data - reconstruction), axis=(1,2,3)
            )
        )
        # ----- KL Divergence ----- #
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var
                - tf.square(z_mean)
                - tf.exp(z_log_var),
                axis=1
            )
        )
        total_loss = recon_loss + kl_loss
        return total_loss, recon_loss, kl_loss

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total_loss, recon_loss, kl_loss = self.compute_losses(data)
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # ----- Update Metrics ----- #
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        total_loss, recon_loss, kl_loss = self.compute_losses(data)
        # ----- Update Metrics ----- #
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs, training=False)
        return self.decoder(z, training=False)

# ========== Re-instantiate and Compile ========== #
vae = VAE(encoder, decoder)
vae.compile(
    optimizer=keras.optimizers.Adam(),
    run_eagerly=True
)

With reference to the Code Cell above, we are able to determine the VAE Model have been successfulyl defined and we are able to proceed to conduct the Training of the VAE in the next sub-section.

---
### 6.3.5 VAE Training

In this sub-section, we will be conducting the VAE Training so as to serve as a comparison between the best GAN Model due to both VAE and GAN being Generative-based Model. The Training will be conducted in the Code Cell below.

In [ ]:
# ========== Data Logger ========== #
csv_logger = CSVLogger('vae_training_log.csv', separator=',', append=False)

# ========== Data Pre-Process ========== #
train_ds = tf.data.Dataset.from_tensor_slices(X_train).shuffle(10_000).batch(128)
val_ds   = tf.data.Dataset.from_tensor_slices(X_val).batch(128)

# ========== VAE Model Training ========== #
history = vae.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[csv_logger]
)

With reference to the output of the Code Cell above, we are able to determine that the VAE have been trained successfully.

---
### 6.3.6 VAE Training Curve Evaluation

In this sub-section, we will be plotting the Learning Curve to assess the Learning Behaviour of the VAE. This plot allows us to visually determine if the VAE is Underfitting, Overfitting, or achieving an Ideal Fit during training. While the VAE is an unsupervised generative model, its ability to minimise both the reconstruction loss and KL divergence effectively indicates how well it learns the latent structure of the input data.

If the VAE is **Underfitting**, it implies that the encoder-decoder pair is not effectively capturing the underlying data distribution or reconstructing input images accurately.

If the VAE is **Overfitting**, the model may have memorised training examples and fails to generalise well to unseen variations in character structures.

An **Ideal Fit** suggests that the VAE is generalising well, balancing both reconstruction quality and latent space regularisation.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Learning Curve**
- Re-tune latent dimensionality or layer complexity
- Increase the number of epochs or reduce dropout
- Revisit the KL-divergence weighting (if applicable)

**Overfitting Learning Curve**
- Enable or increase dropout
- Apply stronger regularisation (e.g. L2)
- Reduce model complexity or batch size

**Ideal Fit Learning Curve**
- Retain the current architecture and hyperparameters
- Proceed with synthetic sample generation per class
- Evaluate generation quality across all 16 classes

With the potential observations and their respective action plans indicated, we will proceed to visualise the Learning Curve in the Code Cell below.

In [ ]:
# ========== Extract Loss Values ========== #
loss = history.history['loss']
val_loss = history.history['val_loss']

# ========== Plot Loss Values ========== #
plt.figure()
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.legend()
plt.title('VAE Total Loss Curve')
plt.show()

With reference to the output of the Learning Curve of the VAE above, we are able to make the following observations:

- No overfitting or underfitting observed, implying model generalises well.

- Loss convergence is stable, with validation closely tracking training.

- VAE training appears healthy, with room for additional fine-tuning or early stopping around epoch 90+.

With the observations indicated above, we will proceed to conduct the Metrics Comparison between the VAE and cDRAGAN in the next sub-section.

---
### 6.3.7 VAE and cDRAGAN Metrics Comparison

In this sub-section, we will be comparing the performance of the VAE and cDRAGAN to test the capabilities of both of the Generative Model. Subseqeuntly, we will evaluate and provide potential reasoning on why is one of the Model more suprerior than the other. The relevant mathematical formulas for the Quantitative Metrics have been indicated below:

---
**Fréchet Inception Distance**

Purpose: Measures the Distance between the Real-Image and Generated-Image Distributions in Inception Feature Space.

$$
\mathrm{FID} \;=\;\|\mu_r - \mu_f\|^2
\;+\;\mathrm{Tr}\Bigl(\Sigma_r + \Sigma_f - 2\,(\Sigma_r\,\Sigma_f)^{\tfrac12}\Bigr)
$$


Where:
- $\mu_r = \mathbb{E}[f(x)]$, $\Sigma_r = \mathrm{Cov}[f(x)]$ for Real Images $x$.  
- $\mu_f = \mathbb{E}[f(\hat x)]$, $\Sigma_f = \mathrm{Cov}[f(\hat x)]$ for Generated Images $\hat x$.  
- $f(\cdot)$ = Map from Image to its InceptionV3 'pooling=avg' Features.  

---
**Diversity (t-SNE Spread)**

Purpose: Quantifies how 'wide' the 2D t-SNE Embedding of Generated Samples.

$$
\mathrm{Spread}
\;=\;
\bigl(\max_i\,z_i^{(1)} - \min_i\,z_i^{(1)}\bigr)
\;\times\;
\bigl(\max_i\,z_i^{(2)} - \min_i\,z_i^{(2)}\bigr)
$$

Where:
- $z_i = (z_i^{(1)}, z_i^{(2)})$ = 2-dimensional t-SNE Embedding of $i$th Generated Image.

---
**Mode Collapse Risk**

Purpose: Flags when too many Generated Images are Nearly Identical (mode collapse).

$$
\text{ModeCollapseRisk} =
\begin{cases}
\text{Low}, & \dfrac{\bigl|\{\mathrm{unique\_rounded}(x_i)\}\bigr|}{N} > \tau,\\
\text{High}, & \text{otherwise}.
\end{cases}
$$

Where:
- $x_i$ = $N$ Generated Samples.  
- $\mathrm{unique\_rounded}(x_i)$ = Rounds Pixels to Detect Duplicates.  
- $\tau=0.9$ = Uniqueness Threshold (90%).

---
**Perceptual Path Length**

Purpose: Measures Sensitively of Generator's Outputs (in VGG16 Feature Space) move when the Latent Code $z$ is Perturbed.

$$
\mathrm{PPL}
\;=\;
\mathbb{E}_{z,\delta z}\Bigl[\,
\|\phi\bigl(G(z + \epsilon\,\delta z)\bigr)\;-\;\phi\bigl(G(z)\bigr)\|_2^2
\Bigr]
$$

Where:  
- $G(z)$ = GAN Generator Mapping $z\in\mathbb{R}^{\mathrm{latent\_dim}}$ to an Image.  
- $\phi(\cdot)$ = VGG16 'pooling=avg' Feature Extractor.  
- $\delta z\sim\mathcal{N}(0,I)$, $\epsilon$ = Small Constant.

---
**Kernel Inception Distance**

Purpose: An Unbiased Estimator of the Squared Maximum Mean Discrepancy (MMD) between Real and Generated Inception Features.

$$
\mathrm{KID}
\;=\;
\frac{1}{m(m-1)}\sum_{i\neq j} k\bigl(\phi(x_i),\phi(x_j)\bigr)
\;+\;
\frac{1}{n(n-1)}\sum_{i\neq j} k\bigl(\phi(\hat x_i),\phi(\hat x_j)\bigr)
\;-\;
\frac{2}{mn}\sum_{i=1}^m\sum_{j=1}^n k\bigl(\phi(x_i),\phi(\hat x_j)\bigr)
$$

Where:  
- $x_i$ ($i=1\ldots m$) = Real Images.
- $\hat x_j$ ($j=1\ldots n$) = Generated Images.  
- $\phi(\cdot)$ = InceptionV3 Feature Extractor (pooling='avg').  
- $k(u,v) = \bigl(\frac{u^\top v}{d}+1\bigr)^3$ = degree-3 Polynomial Kernel on $d$-dimensional Features.

---

With the mathematical and metrics indicated above, we will proceed to conduct the Quantitave Metrics Evalution in the Code Cell below.

In [ ]:
# ========== Evaluate GAN (cDRAGAN) ==========
def evaluate_gan_model(generator, latent_dim, num_classes, X_val,
                       gen_loss_history, disc_loss_history, model_name="cDRAGAN"):
    # ----- Generate Fake Images -----
    n_samples = 1000
    noise = tf.random.normal([n_samples, latent_dim])
    rnd_classes = tf.random.uniform([n_samples], 0, num_classes, dtype=tf.int32)
    fake_labels = tf.one_hot(rnd_classes, depth=num_classes)
    fake_images = generator([noise, fake_labels], training=False)

    # ----- Real Images -----
    real_images = tf.convert_to_tensor(X_val[:n_samples], dtype=tf.float32)

    # ----- FID & KID -----
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)

    # ----- t-SNE Spread -----
    fake_np = ((fake_images + 1.0) / 2.0).numpy().reshape(n_samples, -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(fake_np)
    spread_x = np.ptp(proj[:,0])
    spread_y = np.ptp(proj[:,1])
    tsne_spread = round(spread_x * spread_y, 2)

    # ----- Mode Collapse Risk -----
    unique = np.unique(np.round(fake_np, 3), axis=0).shape[0]
    collapse_risk = "Low" if unique > 0.9 * n_samples else "High"

    # ----- Perceptual Path Length (PPL) -----
    distances = []
    eps = 1e-2
    for _ in range(50):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + eps * tf.random.normal([1, latent_dim])
        img1 = generator([z1, tf.one_hot([0], num_classes)], training=False)
        img2 = generator([z2, tf.one_hot([0], num_classes)], training=False)
        p1 = tf.image.resize(tf.image.grayscale_to_rgb((img1+1)*127.5), (128,128))
        p2 = tf.image.resize(tf.image.grayscale_to_rgb((img2+1)*127.5), (128,128))
        f1 = vgg_model(vgg_preprocess(p1))
        f2 = vgg_model(vgg_preprocess(p2))
        distances.append(tf.reduce_mean(tf.square(f1 - f2)).numpy())
    ppl_value = round(np.mean(distances), 4)

    # ----- Loss Statistics -----
    mean_g = round(np.mean(gen_loss_history), 4)
    std_g  = round(np.std(gen_loss_history), 4)
    mean_d = round(np.mean(disc_loss_history), 4)
    std_d  = round(np.std(disc_loss_history), 4)

    # ----- Assemble DataFrame -----
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_g,
        "Std Generator Loss": std_g,
        "Mean Discriminator Loss": mean_d,
        "Std Discriminator Loss": std_d,
        "Mode Collapse Risk": collapse_risk,
        "Diversity (t-SNE Spread)": tsne_spread,
        "PPL": ppl_value
    }
    return pd.DataFrame([row])

# ========== Evaluate VAE ==========
def evaluate_vae_model(decoder, latent_dim, X_val, history, model_name="VAE"):
    # ----- Generate Fake Images -----
    n_samples = 1000
    noise = tf.random.normal([n_samples, latent_dim])
    fake_images = decoder(noise, training=False)

    # ----- Real Images -----
    real_images = tf.convert_to_tensor(X_val[:n_samples], dtype=tf.float32)

    # ----- FID & KID -----
    fid_value = calculate_fid(real_images, fake_images)
    kid_value = calculate_kid(real_images, fake_images)

    # ----- t-SNE Spread -----
    fake_np = ((fake_images + 1.0) / 2.0).numpy().reshape(n_samples, -1)
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(fake_np)
    spread_x = np.ptp(proj[:,0])
    spread_y = np.ptp(proj[:,1])
    tsne_spread = round(spread_x * spread_y, 2)

    # ----- Mode Collapse Risk -----
    unique = np.unique(np.round(fake_np, 3), axis=0).shape[0]
    collapse_risk = "Low" if unique > 0.9 * n_samples else "High"

    # ----- Perceptual Path Length (PPL) -----
    distances = []
    eps = 1e-2
    for _ in range(50):
        z1 = tf.random.normal([1, latent_dim])
        z2 = z1 + eps * tf.random.normal([1, latent_dim])
        img1 = decoder(z1, training=False)
        img2 = decoder(z2, training=False)
        p1 = tf.image.resize(tf.image.grayscale_to_rgb((img1+1)*127.5), (128,128))
        p2 = tf.image.resize(tf.image.grayscale_to_rgb((img2+1)*127.5), (128,128))
        f1 = vgg_model(vgg_preprocess(p1))
        f2 = vgg_model(vgg_preprocess(p2))
        distances.append(tf.reduce_mean(tf.square(f1 - f2)).numpy())
    ppl_value = round(np.mean(distances), 4)

    # ----- Loss Stats -----
    recon_hist = history.history['recon_loss']
    kl_hist    = history.history['kl_loss']
    mean_recon = round(np.mean(recon_hist), 4)
    std_recon  = round(np.std(recon_hist), 4)
    mean_kl    = round(np.mean(kl_hist), 4)
    std_kl     = round(np.std(kl_hist), 4)

    # ----- Assemble DataFrame -----
    row = {
        "Model": model_name,
        "FID Score": fid_value,
        "KID Score": kid_value,
        "Mean Generator Loss": mean_recon,
        "Std Generator Loss": std_recon,
        "Mean Discriminator Loss": mean_kl,
        "Std Discriminator Loss": std_kl,
        "Mode Collapse Risk": collapse_risk,
        "Diversity (t-SNE Spread)": tsne_spread,
        "PPL": ppl_value
    }
    return pd.DataFrame([row])

# ========== Run & Compare ==========
df_gan = evaluate_gan_model(
    generator=generator,
    latent_dim=160,
    num_classes=16,
    X_val=X_val,
    gen_loss_history=gen_loss_history,
    disc_loss_history=disc_loss_history,
    model_name="cDRAGAN"
)

df_vae = evaluate_vae_model(
    decoder=vae.decoder,
    latent_dim=latent_dim,
    X_val=X_val,
    history=history,
    model_name="VAE"
)

df_compare = pd.concat([df_gan, df_vae], ignore_index=True)
df_compare.style.background_gradient(cmap="Blues")

With reference to the DataFrame above, we are able to make the following implications and reasoning below:

- cDRAGAN is superior in visual fidelity, diversity, and adversarial loss metrics

- VAE performs surprisingly well given its non-adversarial nature

- VAE may be preferable in applications when Structured latent space or stability without adversarial instability is needed.

- cDRAGAN benefits from adversarial training and gradient regularisation, leading to Better FID/KID (image quality), More expressive generator, and Stronger diversity without mode collapse.

- VAE, though not competitive in image realism, is More robust, with stable convergence and low tuning sensitivity and Better suited for tasks like reconstruction, anomaly detection, or latent representation learning.

With the implications and reasoning indicated above, we will proceed to conduct EDAs on our final GAN Model for submission.